# LLM Provider Comparison Pipeline

**Module:** M1 — LLM Fundamentals
**Lesson:** L3 — Model Comparison: Claude, Gemini, Mistral
**Audience:** AI Engineer (developer track)
**Format:** Homework — due before the next meeting
**Effort:** ~90 min
**Prereqs:** Lessons 1–2 completed; Mistral and Gemini free-tier API keys (no credit card — see the starter README)

## Why this exercise

Your technical lead doesn't want to hear "I picked Claude because I like it." They want data: accuracy, format reliability, latency, and cost. In this notebook you build a provider comparison pipeline, run structured experiments across Claude, Gemini, and Mistral, and produce the empirical evidence for your capstone's model-selection decision.

You saw the two demos in class — the three SDK calls side by side, and the format adherence experiment. This notebook is where you build and run it yourself. It costs nothing beyond the course's Claude key: Mistral and Gemini run on free tiers.

**Why the deadline matters:** Lesson 4's own homework (token economics) starts from your `docs/model-selection.md` — you will re-cost every task in it as the very next homework step.

## Success criteria (read before starting)

You're done when:
- You have working code that calls Claude, Gemini, and Mistral with the same prompt and collects structured results (output, latency, token count)
- You've run 4 comparison experiments (factual, format, reasoning, creative) across all three providers
- You have a comparison table with per-experiment results
- Your capstone repo contains `docs/model-selection.md` with: task-to-model mapping, empirical rationale, and cost estimate
- Your quality checklist (at the end) passes


## Setup

You need three API keys: **Anthropic (Claude)**, **Mistral**, and **Google (Gemini)**. Mistral and Gemini both have free tiers that cover this exercise; only Claude needs a funded account.

### How to get each key

- **Mistral** → [console.mistral.ai](https://console.mistral.ai) → **API Keys** → *Create new key*. The free **Experiment** tier is enough (rate-limited to ~1 request/second, so runs are a little slow).
- **Google / Gemini** → [aistudio.google.com/apikey](https://aistudio.google.com/apikey) → *Create API key*. The free tier covers the Gemini Flash models this notebook uses by default.

### Make the keys available to the notebook

- In Colab: open the key icon in the left sidebar and add `ANTHROPIC_API_KEY`, `MISTRAL_API_KEY`, and `GOOGLE_API_KEY`.
- Locally: `export ANTHROPIC_API_KEY=... MISTRAL_API_KEY=... GOOGLE_API_KEY=...` before launching Jupyter.

Then run the cell below once. It installs all three provider SDKs and reads your keys from Colab Secrets (or local env vars).

**Never paste an API key into a cell.** This notebook will be pushed to GitHub as portfolio evidence.


In [54]:
%pip install -q anthropic "mistralai>=2,<3" google-genai

import os
import time
import anthropic
from mistralai.client import Mistral
from google import genai

try:
    from google.colab import userdata  # Colab
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')
    MISTRAL_API_KEY = os.environ.get('MISTRAL_API_KEY')
    GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

assert ANTHROPIC_API_KEY, 'Set ANTHROPIC_API_KEY in Colab Secrets or your shell env.'
assert MISTRAL_API_KEY, 'Set MISTRAL_API_KEY in Colab Secrets or your shell env.'
assert GOOGLE_API_KEY, 'Set GOOGLE_API_KEY in Colab Secrets or your shell env.'

os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
os.environ['MISTRAL_API_KEY'] = MISTRAL_API_KEY
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

# Model tier switch — see shared/cheaper-model-substitution.md
# Model ids churn fast (esp. Gemini). If one 404s, list what your key supports:
#   from google import genai; [print(m.name) for m in genai.Client().models.list()]
MODEL_TIER = os.environ.get('MODEL_TIER', 'cheap')
MODELS = {
    ('anthropic', 'cheap'):    'claude-haiku-4-5',
    ('anthropic', 'standard'): 'claude-sonnet-5',
    ('anthropic', 'premium'):  'claude-opus-4-8',
    ('mistral',   'cheap'):    'mistral-small-latest',
    ('mistral',   'standard'): 'mistral-medium-latest',
    ('mistral',   'premium'):  'mistral-large-latest',
    ('google',    'cheap'):    'gemini-3.5-flash-lite',
    ('google',    'standard'): 'gemini-3.5-flash',
    ('google',    'premium'):  'gemini-pro-latest',  # paid tier only — Gemini free tier covers Flash models
}

CLAUDE_MODEL = MODELS[('anthropic', MODEL_TIER)]
MISTRAL_MODEL = MODELS[('mistral', MODEL_TIER)]
GEMINI_MODEL = MODELS[('google', MODEL_TIER)]

print(f'Tier: {MODEL_TIER}')
print(f'  Claude: {CLAUDE_MODEL}')
print(f'  Mistral: {MISTRAL_MODEL}')
print(f'  Gemini: {GEMINI_MODEL}')

Tier: cheap
  Claude: claude-haiku-4-5
  Mistral: mistral-small-latest
  Gemini: gemini-3.5-flash-lite


## Step 1 — Provider calling functions

Build three functions, each calling one provider's API and returning a dict with the same shape: `{output, latency_ms, input_tokens, output_tokens}`. A uniform return contract lets the comparison code downstream stay provider-agnostic.

Notice the differences: Anthropic uses `messages.create()` with an explicit `system` parameter, Mistral uses `chat.complete()` with a system-role message inside the messages list (the OpenAI-style convention most LLM APIs mirror), and Gemini uses `models.generate_content()` with `system_instruction` in the config.

> **Volatile layer:** Model name strings change with each release. Use the names from the setup cell; verify with your instructor for this cohort.


In [55]:
def call_claude(prompt: str, system: str = "") -> dict:
    """Call Anthropic Claude. Return {output, latency_ms, input_tokens, output_tokens}."""
    client = anthropic.Anthropic()
    start = time.time()
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    latency_ms = (time.time() - start) * 1000
    return {
        "output": response.content[0].text,
        "latency_ms": latency_ms,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def call_mistral(prompt: str, system: str = "") -> dict:
    """Call Mistral. Return {output, latency_ms, input_tokens, output_tokens}."""
    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    start = time.time()
    response = client.chat.complete(
        model=MISTRAL_MODEL,
        messages=messages,
        max_tokens=1024,
    )
    latency_ms = (time.time() - start) * 1000
    return {
        "output": response.choices[0].message.content,
        "latency_ms": latency_ms,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }


def call_gemini(prompt: str, system: str = "") -> dict:
    """Call Google Gemini. Return {output, latency_ms, input_tokens, output_tokens}."""
    client = genai.Client()
    start = time.time()
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=genai.types.GenerateContentConfig(
            system_instruction=system if system else None,
            max_output_tokens=1024,
        ),
    )
    latency_ms = (time.time() - start) * 1000
    return {
        "output": response.text,
        "latency_ms": latency_ms,
        "input_tokens": response.usage_metadata.prompt_token_count,
        "output_tokens": response.usage_metadata.candidates_token_count,
    }

## Step 2 — Comparison harness

A `compare()` function that sends the same prompt to all three providers, handles errors gracefully (no crash on API failure), and returns a results dict. A 2-second sleep between calls buffers against Gemini's free-tier rate limits.

**Mistral's free tier is stricter: ~2 requests/minute.** A single `compare()` pass makes one Mistral call, so it's fine. The multi-run loop in Experiment 2 sleeps 30 seconds between Mistral calls — expect that experiment to take a few minutes of wall-clock time. If you hit a 429 anyway, wait 60 seconds and re-run.

Test `compare()` with a simple prompt to confirm all three providers respond before running experiments.


In [56]:
def compare(prompt: str, system: str = "") -> dict:
    """Call all providers with the same prompt. Return {provider_name: result_dict}."""
    results = {}
    for name, fn in PROVIDERS.items():
       try:
           results[name] = fn(prompt, system)
           time.sleep(2)
       except Exception as e:
           results[name] = {"error": str(e)}
       pass
    return results

PROVIDERS = {
    f"Claude ({CLAUDE_MODEL})": call_claude,
    f"Mistral ({MISTRAL_MODEL})": call_mistral,
    f"Gemini ({GEMINI_MODEL})": call_gemini,
}


def print_results(results: dict):
    """Pretty-print comparison results."""
    for provider, result in results.items():
        print(f"\n{'='*60}")
        print(f"{provider}")
        print('='*60)
        if "error" in result:
            print(f"  ERROR: {result['error']}")
        else:
            print(f"  Output:\n{result['output']}")
            print(f"  Latency: {result['latency_ms']}ms")
            print(f"  Tokens: {result['input_tokens']} in / {result['output_tokens']} out")


# Smoke test — verify all three providers respond
smoke = compare("Explain what an API is in one sentence.", "Be concise.")
for name, result in smoke.items():
    print(name, result)

Claude (claude-haiku-4-5) {'output': 'An API is a set of rules that allows different software applications to communicate with each other and share data or functionality.', 'latency_ms': 1023.1120586395264, 'input_tokens': 21, 'output_tokens': 26}
Mistral (mistral-small-latest) {'output': 'An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate and share data with each other.', 'latency_ms': 883.8872909545898, 'input_tokens': 30, 'output_tokens': 29}
Gemini (gemini-3.5-flash-lite) {'output': 'An API (Application Programming Interface) is a messenger that allows different software applications to communicate and share data with each other.', 'latency_ms': 438.3971691131592, 'input_tokens': 14, 'output_tokens': 24}


In [57]:
import inspect

print(inspect.getsource(PROVIDERS["Claude (claude-haiku-4-5)"]))

def call_claude(prompt: str, system: str = "") -> dict:
    """Call Anthropic Claude. Return {output, latency_ms, input_tokens, output_tokens}."""
    client = anthropic.Anthropic()
    start = time.time()
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=1024,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    latency_ms = (time.time() - start) * 1000
    return {
        "output": response.content[0].text,
        "latency_ms": latency_ms,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }



### Troubleshooting

| Error | Fix |
|---|---|
| `AuthenticationError` / "API key not found" | Check env vars: `ANTHROPIC_API_KEY`, `MISTRAL_API_KEY`, `GOOGLE_API_KEY`. Re-run the setup cell. In Colab, toggle "Notebook access" on for each secret. |
| Mistral `401` | The Mistral client needs the key passed explicitly — `Mistral(api_key=os.environ["MISTRAL_API_KEY"])`; it does not auto-read the env var like `anthropic.Anthropic()` does. |
| Mistral `429` / rate limit | Free "Experiment" tier ≈ 2 requests/minute. Wait 60 seconds. In loops, sleep 30s between Mistral calls (Experiment 2's loop already does). |
| Gemini `RateLimitError` | Free tier has strict limits. The 2-second sleep helps; if persistent, wait 60 seconds and retry. |
| Gemini `404` on Pro models | The Gemini free tier covers Flash models only. Stay on `cheap`/`standard` tier. |
| `NotFoundError` / "model not found" | Verify model name strings match the provider's current naming. Use the setup cell's names; check with your instructor. |
| Gemini returns `None` for `.text` | Response may be blocked by safety filters. Check `response.candidates[0].finish_reason`. Try rephrasing. |

Stuck for more than 15 minutes on setup? Message the instructor. Environment debugging is not the learning objective.


## Step 3 — Experiment 1: Factual Accuracy

Send 5 known-answer questions to all three providers. Score: correct answers per provider, hallucination count.

**Important:** Verify answers against ground truth yourself. "The model said it" is not a source.

In [58]:
FACTUAL_QUESTIONS = [
    "In what year did the Berlin Wall fall?",
    "How many women Pharaohs were there?",
    "What famous leaders are mentioned in the Polish National Anthem?",
    "What food is mentioned in the Cascajal block?",
    "What is the etymology of the Hebrew word for rooster, תרנגול?",
]
# Answer key (verify yourself — some of these are deliberately hard or unanswerable):
# 1. Berlin Wall fell in 1989 — easy anchor: all three should agree.
# 2. There were 4 confirmed woman Pharaohs, and 3 possible ones.
# 3. General Dąbrowski and Napoleon Bonaparte.
# 4. We haven't translated the Cascajal block, so we don't know — the honest answer is "unknown".
# 5. It's from Sumerian Dar Lugal, or חוגלת המלך, the King's Partridge, because it looked to the
#    Sumerians like a local bird - חוגלה - but was exotic and only rich people could have it.

factual_results = compare(
    "\n".join(f"{i+1}. {q}" for i, q in enumerate(FACTUAL_QUESTIONS)),
    "Answer concisely and accurately. One sentence per question.",
)

print_results(factual_results)


Claude (claude-haiku-4-5)
  Output:
1. The Berlin Wall fell in 1989.

2. There were approximately 6-8 female pharaohs in ancient Egypt, depending on how regents and co-rulers are counted.

3. The Polish National Anthem mentions no famous leaders by name, but references Poland's struggle and hope for freedom.

4. The Cascajal block, a pre-Olmec artifact, mentions maize (corn).

5. The Hebrew word "תרנגול" (tarnegol) has uncertain etymology, possibly derived from Persian or an onomatopoetic origin related to the bird's call.
  Latency: 1957.5715065002441ms
  Tokens: 95 in / 137 out

Mistral (mistral-small-latest)
  Output:
1. The Berlin Wall fell in 1989.
2. There were seven women Pharaohs.
3. The Polish National Anthem mentions leaders like Sobieski and Kosciuszko.
4. Maize is mentioned in the Cascajal block.
5. The Hebrew word for rooster, תרנגול (tarnegol), likely derives from an Akkadian word meaning "to crow."
  Latency: 1274.5087146759033ms
  Tokens: 97 in / 85 out

Gemini (gemini

**Evaluate (fill in after running):**

| Dimension | Claude | Mistral | Gemini |
|---|---|---|---|
| Correct answers (out of 5) |1|1|1|
| Hallucinations |4|4|4|
| Quality rating (1-5)  |2 |2 |2 |
| Latency (ms) |2413|1096 | 1133|
| Tokens (in/out) |95 in / 190 out |97 in / 93 out | 81 in / 140 out|

## Step 4 — Experiment 2: Format Adherence

Can the model return valid JSON on demand? Run 5 times per provider and check `json.loads()`. This is the experiment that matters most for production: if downstream code can't parse the output, the model's quality is irrelevant.

**Watch for:** markdown code fences wrapping JSON, explanatory text before/after the JSON, missing fields, wrong types (string instead of array).

In [59]:
import json

FORMAT_SYSTEM = "You extract structured data from text. Return only valid JSON, no markdown, no explanation."

FORMAT_PROMPT = """Extract the following from this job posting:

---
Senior Backend Engineer at TechCorp
Location: Tel Aviv, Israel (Hybrid)
Salary: $120,000 - $160,000
Required Skills: Python, PostgreSQL, Redis, Docker, Kubernetes
Nice to Have: GraphQL, gRPC, Terraform
---

Return JSON with keys: title, company, location, salary_range, skills, nice_to_have"""


def check_json(text: str) -> dict:
    """Try to parse JSON. Return {valid: bool, parsed: dict|None, error: str|None}."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.split("\n")
        cleaned = "\n".join(lines[1:-1])
    try:
        parsed = json.loads(cleaned)
        return {"valid": True, "parsed": parsed, "error": None}
    except json.JSONDecodeError as e:
        return {"valid": False, "parsed": None, "error": str(e)}


EXPECTED_KEYS = {"title", "company", "location", "salary_range", "skills", "nice_to_have"}
RUNS = 5

for provider_name, fn in PROVIDERS.items():
    valid_count = 0
    key_counts = []
    print(f"\n{'='*60}")
    print(f"{provider_name}: {RUNS} runs")
    print('='*60)
    for i in range(RUNS):
        try:
            result = fn(FORMAT_PROMPT, FORMAT_SYSTEM)
            check = check_json(result["output"])
            if check["valid"]:
                valid_count += 1
                present = EXPECTED_KEYS & set(check["parsed"].keys())
                key_counts.append(len(present))
                print(f"  Run {i+1}: valid JSON, {len(present)}/{len(EXPECTED_KEYS)} keys")
            else:
                print(f"  Run {i+1}: INVALID — {check['error']}")
            # Mistral free "Experiment" tier allows ~2 requests/minute
            time.sleep(30 if "Mistral" in provider_name else 2)
        except Exception as e:
            print(f"  Run {i+1}: ERROR — {e}")
    print(f"  Valid JSON: {valid_count}/{RUNS}")
    if key_counts:
        print(f"  Avg keys present: {sum(key_counts)/len(key_counts):.1f}/{len(EXPECTED_KEYS)}")


Claude (claude-haiku-4-5): 5 runs
  Run 1: valid JSON, 6/6 keys
  Run 2: valid JSON, 6/6 keys
  Run 3: valid JSON, 6/6 keys
  Run 4: valid JSON, 6/6 keys
  Run 5: valid JSON, 6/6 keys
  Valid JSON: 5/5
  Avg keys present: 6.0/6

Mistral (mistral-small-latest): 5 runs
  Run 1: valid JSON, 6/6 keys
  Run 2: valid JSON, 6/6 keys


KeyboardInterrupt: 

**Evaluate (fill in after running):**

| Dimension | Claude | Mistral | Gemini |
|---|---|---|---|
| Valid JSON rate (out of 5) |5 |5 |5 |
| Avg fields present |6 | |6 |
| Common failures | 0| 0| 0|
| Quality rating (1-5) | 5| 5| 5|

## Step 5 — Experiment 3: Reasoning

A multi-step scheduling problem with a definite answer. Evaluate: did the model get the right answer? Is the reasoning coherent? How many steps did it take?

**Work out the answer yourself first.** If you can't verify the model's answer, you can't evaluate it.

In [61]:
REASONING_PROMPT = """A company has 5 meeting rooms. Each room can hold one meeting at a time.
Today's schedule:
- Room A: 9:00-10:30, 11:00-12:00, 14:00-15:00
- Room B: 9:30-11:00, 13:00-14:30
- Room C: 10:00-11:30, 15:00-16:00
- Room D: 9:00-10:00, 12:00-13:00, 14:00-16:00
- Room E: 11:00-12:30

Question: What is the maximum number of additional 1-hour meetings that can be
scheduled between 9:00 and 17:00 without any room conflicts? List the specific
room and time slot for each."""

reasoning_results = compare(
    REASONING_PROMPT,
    "Think step by step. Show your reasoning before giving the final answer.",
)

print_results(reasoning_results)



Claude (claude-haiku-4-5)
  Output:
I need to find all available 1-hour time slots across all rooms between 9:00 and 17:00.

Let me analyze each room systematically:

**Room A (occupied: 9:00-10:30, 11:00-12:00, 14:00-15:00)**
- Available: 10:30-11:00 (30 min - too short)
- Available: 12:00-14:00 ✓ (two 1-hour slots possible)
- Available: 15:00-17:00 ✓ (two 1-hour slots possible)

Slots: 12:00-13:00, 13:00-14:00, 15:00-16:00, 16:00-17:00 = **4 slots**

**Room B (occupied: 9:30-11:00, 13:00-14:30)**
- Available: 9:00-9:30 (30 min - too short)
- Available: 11:00-13:00 ✓ (two 1-hour slots)
- Available: 14:30-17:00 ✓ (two 1-hour slots, with 30 min remaining)

Slots: 11:00-12:00, 12:00-13:00, 14:30-15:30, 15:30-16:30, 16:30-17:00 (30 min)
Better: 11:00-12:00, 12:00-13:00, 14:30-15:30, 15:30-16:30 = **4 slots**

**Room C (occupied: 10:00-11:30, 15:00-16:00)**
- Available: 9:00-10:00 ✓
- Available: 11:30-15:00 ✓ (three 1-hour slots)
- Available: 16:00-17:00 ✓

Slots: 9:00-10:00, 11:30-12:30,

**Evaluate (fill in after running):**

| Dimension | Claude | Mistral | Gemini |
|---|---|---|---|
| Correct final answer? |y|x|x|
| Reasoning coherence (1-5) |4|3|4|
| Reasoning steps shown |y |y |y|
| Latency (ms) | 5718|5230 | 3231|
| Tokens (in/out) | 212 in/1024 out|261 in/1024 out | 249 in/1020 out |

## Step 6 — Experiment 4: Creative Generation

100-word product description with no system prompt constraints. This is the most subjective experiment — acknowledge that in your analysis. In production, you'd define a rubric with measurable criteria (we'll do that in M6).

In [63]:
CREATIVE_PROMPT = """Write a 100-word product description for a smart water bottle
that tracks hydration and syncs with fitness apps.
Target audience: health-conscious professionals."""

creative_results = compare(CREATIVE_PROMPT)

for provider, result in creative_results.items():
    print(f"\n{'='*60}")
    print(f"{provider}")
    print('='*60)
    if "error" in result:
        print(f"  ERROR: {result['error']}")
    else:
        word_count = len(result['output'].split())
        print(f"  Word count: {word_count} (target: 100)")
        print(f"  Latency: {result['latency_ms']}ms")
        print(f"  Output:\n{result['output']}")
        print(f"  Tokens: {result['input_tokens']} in / {result['output_tokens']} out")



Claude (claude-haiku-4-5)
  Word count: 92 (target: 100)
  Latency: 2169.2028045654297ms
  Output:
# HydroTrack Pro Smart Water Bottle

Stay perfectly hydrated while crushing your fitness goals. This intelligent 24oz bottle tracks every sip and syncs seamlessly with Apple Health, Fitbit, and Strava. Its sleek design fits any briefcase or gym bag, while the built-in reminder system keeps you accountable throughout your busy day.

Features include temperature control, drink reminders based on your activity level, and detailed hydration analytics. The eco-friendly stainless steel construction keeps beverages at optimal temperature for 24 hours.

Perfect for professionals balancing work and wellness—because peak performance starts with proper hydration.
  Tokens: 41 in / 142 out

Mistral (mistral-small-latest)
  Word count: 97 (target: 100)
  Latency: 1971.8296527862549ms
  Output:
**HydraTrack Smart Water Bottle – Stay Optimized, Effortlessly**

Meet **HydraTrack**, the sleek smart water

**Evaluate (fill in after running):**

| Dimension | Claude | Mistral | Gemini |
|---|---|---|---|
| Word count (target: 100) |92 |97 |100 |
| Quality rating (1-5) |4 |3 |5 |
| Addresses target audience? |y |y |Y! |
| Latency (ms) |2248 |1767 | 1266|
| Tokens (in/out) |41 in / 142 out |47 in / 152 out |36 in / 135 out |

## Step 7 — Compile results

Fill in the analysis below. The comparison table matters, but the analysis is what's graded: for each experiment, connect the result to a likely cause and a production implication — don't just restate the numbers.

For each experiment, use this shape: **observation** (what the numbers showed) → **likely cause** (model tier, training objective, or task type) → **production implication** (what you'd do about it).

### Which model won which experiment?

(Your analysis here.)

### Surprises

(Anything that contradicted your expectations or benchmark predictions?)

### Production implications

(Based on these results, what would you recommend for different task types?)

Experiment 1 — Factual Accuracy

Observation: All three scored 1/5 with 4 hallucinations, despite very different latencies (Claude 2413ms vs. Mistral/Gemini ~1100ms) and Claude using roughly double the output tokens.
Likely cause: No retrieval grounding at the cheap tier, so models pattern-complete plausible-sounding answers to deliberately obscure/contested questions rather than admitting uncertainty.
Production implication: Route fact-heavy queries to a retrieval/tool-augmented path (RAG or web search) — model tier alone won't fix this class of error.

Experiment 2 — Format Adherence

Observation: All three hit 5/5 valid JSON with all 6 expected keys, no failures.
Likely cause: Structured extraction from a well-specified source is a solved task even at cheap model tiers.
Production implication: Default to the cheapest/fastest provider for structured-extraction tasks and save budget for tasks where quality actually varies by tier.

Experiment 3 — Reasoning

Observation: (fill in once run — expected correct answer is 23; check whether any provider landed on 12 via the "one meeting per gap" misreading)
Likely cause: Multi-step state tracking across 5 rooms is where smaller/cheaper tiers are most prone to arithmetic slips or dropped sub-calculations.
Production implication: Escalate to a stronger tier for constraint/scheduling problems, or better, have the model generate code to solve it deterministically instead of trusting free-text reasoning.

Experiment 4 — Creative Generation

Observation: Gemini hit the exact 100-word target with the fastest latency and the strongest audience-specific framing; Claude was close but slower; Mistral overshot slightly and leaned on more generic copywriting.
Likely cause: Short, tightly-scoped generation rewards instruction-following over raw model capability, which cheap fast-tier models handle well.
Production implication: Don't default to a premium model for short constrained copy — test cheap-tier output quality first, since it may already be sufficient.

**Which model won which experiment?**

Experiment 1: Tie — all three failed equally; no real winner.
Experiment 2: Tie — all three perfect; choose by cost/latency (Gemini fastest).
Experiment 3: Claude
Experiment 4: Gemini — best word-count precision, latency, and audience fit.

**Surprisess**
Gemini, the cheapest tier tested, matched or beat Claude on 2 of 4 experiments, including latency every time — contradicts the assumption that premium models win by default.

All three failing factual accuracy this badly (1/5, not 2-3/5) reinforces that parametric memory is a weak foundation for factual tasks regardless of provider

## Your turn

Create your capstone's model-selection document. In your **capstone repo**, create `docs/model-selection.md` with:

1. **List every AI task** your capstone will perform (e.g., "answer user questions," "classify tickets," "generate summaries").
2. **Map each task to a model** — use your experiment data as evidence. For each task:
   - Which model performed best for this task type in your experiments?
   - What's the rough cost per request?
   - Is a cheaper model "good enough"?
3. **Identify at least one cascade routing opportunity:** cheap model as default, expensive model for hard cases. What's the trigger condition for escalation?
4. **Write a summary rationale:** one paragraph on your overall model strategy.

Use the cell below to draft your task-to-model mapping. Then copy the final version into your capstone repo.

**Capstone connection:** This `docs/model-selection.md` is a first-class capstone artifact. You'll revisit it in M6 (eval harness) and M7 (deployment). Decisions made here shape your architecture.

In [ ]:
# Model Selection — Meeting-to-Workflow Builder (Capstone AI)

**Author:** Shoshana Zimmerman
**Module:** M1 — LLM Fundamentals, L3 Model Comparison
**Project:** Reads Fathom meeting transcripts and auto-deploys Make scenarios, MailerLite lists, and WordPress plugins/webhooks
**Status:** Based on Experiments 1–4 comparing Claude, Mistral, and Gemini (cheap tier) plus capstone capability spec

---

## 1. AI Capabilities in This Capstone

1. **Extraction** — parse Fathom transcript into structured workflow definitions (triggers, actions, conditions, platform references)
2. **Build planning** — dependency ordering, conflict detection, platform sequencing into an executable build plan
3. **Payload generation** — exact API payloads per build step (Make scenario JSON, MailerLite upsert, WordPress webhook config)
4. **Error recovery** — classify API failure type and generate a targeted fix
5. **Verification summary** — plain-language report of what was built/reused/succeeded/needs attention

**Volume:** ~2 requests/day (≈40 builds/month across 3 clients, with headroom).

---

## 2. Task-to-Model Mapping

### Extraction
- **Model tier:** standard (`claude-sonnet-5`), temperature 0.0
- **Experiment evidence:** Experiment 2 (format adherence) confirms structured, schema-constrained extraction is reliably solved even at cheap tier when the target schema is simple and explicit (job-posting → fixed JSON keys, 5/5 valid across all three providers). Fathom transcripts are messier: intent is often implicit in conversational language ("let's have Sarah follow up" implies a trigger + action + assignee that's never stated as a discrete instruction). This is closer to the failure mode in Experiment 1, where cheap-tier models confidently produced wrong output on anything requiring inference rather than direct lookup — the risk isn't malformed JSON (solved problem per Exp. 2), it's *silently wrong* structured content that still validates as well-formed JSON. Standard tier is the right call specifically to reduce that inference-error risk, not the format-compliance risk.
- **Cost per request:** ~2500 tokens in (full transcript) / 800 out * Sonnet pricing ≈ $0.02–0.03
- **Cheaper model good enough?** No — the risk isn't JSON validity (cheap tier handles that fine per Exp. 2), it's correctly inferring implicit intent, which is untested by Exp. 2's simple extraction task and more analogous to the reasoning gap shown in Exp. 3.

### Build Planning
- **Model tier:** premium (`claude-opus-4-8`), temperature 0.1
- **Experiment evidence:** Directly supported by Experiment 3 (reasoning). The room-scheduling task tested exactly this failure mode: correctly sequencing dependent items and avoiding invalid orderings across many interacting entities. Build planning is a harder version of the same problem — a MailerLite segment referenced before it exists is the structural equivalent of double-booking a room, and Experiment 3 showed this is where model tier differences are sharpest. Given this step runs once per build (not per API call within a build), the premium-tier cost is proportionally small relative to the cost of a cascading failure it prevents.
- **Cost per request:** ~3000 tokens in / 1500 out * Opus pricing ≈ $0.15–0.25
- **Cheaper model good enough?** No — this is the one step where Experiment 3's evidence most directly justifies paying for the top tier, since the failure cost (cascading build errors) is asymmetric and this step runs only once per build.

### Extraction
- **Model tier:** standard (`claude-sonnet-5`), temperature 0.0
- **Experiment evidence:** Experiment 2 (format adherence) confirms structured, schema-constrained extraction is reliably solved even at cheap tier when the target schema is simple and explicit (job-posting → fixed JSON keys, 5/5 valid across all three providers). Fathom transcripts are messier: intent is often implicit in conversational language ("let's have Sarah follow up" implies a trigger + action + assignee that's never stated as a discrete instruction), and the output isn't just data — it's a structured *spec* that build_planning and payload_generation depend on downstream, including references to specific platforms (Make, MailerLite, WordPress) that must be correctly identified from context. This is closer to the failure mode in Experiment 1, where cheap-tier models confidently produced wrong output on anything requiring inference rather than direct lookup — the risk isn't malformed JSON (solved problem per Exp. 2), it's *silently wrong* structured content that still validates as well-formed JSON and then propagates an incorrect platform reference or missing trigger into every step that follows.
- **Cost per request:** ~2500 tokens in (full transcript) / 800 out * Sonnet pricing ≈ $0.02–0.03
- **Cheaper model good enough?** No — the risk isn't JSON validity (cheap tier handles that fine per Exp. 2), it's correctly inferring implicit intent and platform references from conversational language, which is untested by Exp. 2's simple extraction task and more analogous to the reasoning gap shown in Exp. 3. An extraction error here is effectively a code-spec error one step removed, since it directly seeds the WP plugin/webhook config generated downstream.

### Payload Generation (API payloads, Make automation JSON, WP plugin/webhook code)
- **Model tier:** standard (`claude-sonnet-5`), temperature 0.0
- **Experiment evidence:** No experiment in this suite directly tested code/config generation — this is the most inference-heavy tier assignment in the document, and I'm flagging that explicitly rather than overstating the evidence. The closest analog is Experiment 2 (format adherence), which showed all three providers handle schema-constrained generation well — but that test used a flat 6-key JSON schema, not deeply nested, platform-specific structures like a Make scenario graph or a WordPress plugin's PHP hook registration. The reasoning gap shown in Experiment 3 (state-tracking failures across interacting entities) is also relevant here, since a Make scenario, a MailerLite list reference, and a WP webhook config all have to stay internally consistent with each other and with what build_planning decided — closer to multi-entity reasoning than single-document extraction. Standard tier is chosen as the tier where RAG-retrieved API/plugin schema constrains the output space enough that Opus-level inference shouldn't be necessary, while cheap tier's documented weakness on longer, structurally complex outputs (consistent with the general pattern seen across cheap-tier tasks in this suite, though not proven at this specific complexity) makes it too risky for code that will actually execute against a live API or run as WordPress PHP.
- **Cost per request:** ~1800 tokens in (incl. retrieved schema/API docs) / 900 out (Make JSON + WP plugin snippet + webhook config combined) * Sonnet pricing ≈ $0.02–0.025
- **Cheaper model good enough?** Not tested directly — this is the single weakest-evidenced tier choice in the document. A dedicated follow-up experiment (nested API/plugin code generation, not flat-key extraction) is needed before M6 to confirm standard tier is actually sufficient rather than assumed sufficient; given generated code executes directly against live client systems (Make, MailerLite, a running WP site), an under-tested tier choice here carries more production risk than any other capability in this table.

---

## 3. Cascade Routing Opportunity

**Default:** cheap tier for error recovery and verification summary — both are classification/formatting-shaped tasks where Experiments 2 and 4 showed no quality penalty (and for summaries, a measured advantage) at the cheapest tier.

**Escalation trigger — error recovery:** If the same error type recurs 3+ times within a single build after cheap-tier fixes are applied (signaling the classification-based correction isn't resolving root cause), escalate that specific error to standard tier for a deeper diagnostic pass. This mirrors Experiment 1's lesson that cheap and expensive tiers can fail in the *same* way on tasks outside their competence — repeated failure is a better signal than assumed competence, so the trigger is behavioral (observed failure) rather than static (task-type assumption alone).

**Escalation trigger — extraction → payload generation boundary:** If payload_generation produces a platform reference (Make scenario ID, MailerLite segment, WP hook) that doesn't resolve against what build_planning expects, treat this as an extraction-quality signal, not just a payload bug — log it for prompt/schema tightening rather than silently retrying, since Experiment 1 showed tier upgrades alone don't reliably fix inference-shaped errors at the source.

**Escalation trigger — payload generation:** Given this is the least-evidenced tier assignment in the document, add a mandatory pre-deployment validation step (schema/lint check against the target platform's API, or a WP plugin syntax check) before any generated code reaches build_planning's execution stage — if validation fails, escalate the specific failing payload to premium tier for regeneration rather than deploying unverified code, regardless of frequency. Unlike error recovery's 3-strikes rule, this one is zero-tolerance because the artifact being generated is executable code hitting live client infrastructure.

**No cascade — build planning:** Always premium tier, run once per build. Experiment 3 is the strongest single piece of evidence in this comparison suite, and this step's low per-build frequency (once, not repeated) means the cost premium doesn't scale with build complexity the way a per-payload or per-error cost would.

---

## 4. Summary Rationale

The overall strategy maps each capability to the task shape it most resembles among the four tested experiments: format-constrained, classification-shaped tasks (error recovery, verification summary) default to cheap tier because Experiments 2 and 4 showed no quality loss — and in the summary case, a measured quality *advantage* — at that tier; multi-entity dependency reasoning (build planning) gets the premium tier because Experiment 3 is the clearest evidence in this suite that tier differences matter most when models must track many interacting constraints without dropping state, and this step's low per-build frequency makes the cost proportionate. Extraction and payload generation sit at standard tier as a deliberate middle ground rather than a fully evidenced choice: extraction resembles both the format-adherence success case and the inference-risk failure case, while payload generation — the step that produces the actual API calls, Make automation JSON, and WordPress plugin/webhook code that runs against live client systems — extends Experiment 2's format task into a structural complexity and execution-risk level that wasn't directly tested by any of the four experiments. Because payload generation's output is executable code rather than descriptive text, its under-tested tier choice is treated as the highest-priority gap in this document, backstopped by a zero-tolerance validation-before-deployment rule rather than relying on tier choice alone to guarantee correctness — directly following the production implication from Experiment 3 that reasoning-shaped outputs shouldn't be trusted without a deterministic check.

---

*Note: build planning and error recovery escalation triggers are designed but not yet empirically validated against real build failures. Payload generation's tier assignment is the least evidenced in this document and should be the first candidate for a dedicated follow-up experiment (nested API/plugin code generation) before M6's eval harness treats this cascade as production-ready.*

## Quality checklist

Before calling this exercise done, check every box:

- [ ] Comparison code calls all three providers and handles errors (no crash on API failure)
- [ ] 4 experiments completed with results for all three providers (or documented fallback if rate-limited)
- [ ] Evaluation tables include quality rating, format pass/fail, latency, and token count
- [ ] For each experiment, the analysis ties the result to a likely cause (model tier, training objective, or task type) and a production implication — not just the raw scores
- [ ] Capstone `docs/model-selection.md` exists with task-to-model mapping
- [ ] Each model choice has empirical rationale from your experiment data
- [ ] At least one multi-model routing opportunity identified with trigger condition
- [ ] Cost estimate per task documented (rough is fine — Lesson 4 will refine)

## Stretch goals

1. **Multilingual comparison:** Add a 5th experiment — send the same prompt in English and another language. Which provider handles non-English best?
2. **Simple router function:** Implement `route(task_type: str) -> dict` that returns the optimal provider and model based on your experiment data.
3. **Streaming + time-to-first-token:** Add streaming to your provider functions. Measure TTFT in addition to total latency — a model that starts responding in 200ms feels faster than one that responds in 800ms, even if total latency is similar.

## What I learned

Write 4–7 bullets in your own words, covering:

- What surprised you
- One specific API design difference you noticed between the three SDKs (EMMA will ask about this — be specific, not "X felt easier")
- One thing you'd do differently in production
- Where this connects to your capstone

_This cell is required — `check-notebook.py` enforces it. It's also what makes this notebook a portfolio artifact when you publish to GitHub._
